In [ ]:
import os


project_dir = '/blue/shenhaowang/qingqisong/be-and-active-travel'
os.chdir(project_dir)

In [ ]:
"""
================================================================================
Chicago Baseline Gravity Model for OD Flow Prediction
================================================================================
PURPOSE:
    Transform person-level travel survey data into zone-level OD flows
    and build a baseline gravity model for OD prediction in Chicago.

KEY DIFFERENCES FROM PERSON-LEVEL PIPELINE:
    1. Unit of analysis: Census Block Group (CBG) pairs, not persons
    2. Dependent variable: Trip count between CBG zones, not individual
       travel duration
    3. Independent variables: Zone-level attributes (population density,
       employment density, built environment) + inter-zone distance,
       not person-level demographics
    4. Model: Gravity model fitted via Poisson regression

OVERVIEW:
    Phase 1 - Extract trip-level origin-destination pairs from survey
    Phase 2 - Assign O/D locations to census block groups (spatial zones)
    Phase 3 - Aggregate individual trips into an OD flow matrix
    Phase 4 - Compute zone-level features for origins and destinations
    Phase 5 - Compute inter-zone distances and OD-pair features
    Phase 6 - Fit baseline gravity model and evaluate
================================================================================
"""

import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import KDTree
from shapely.geometry import Point
from datetime import datetime
import warnings
import os
import glob

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """
    Central configuration for all file paths and constants.
    """

    # --- Raw survey data (same as before) ---
    PLACE_DF    = './mydailytravel/place.csv'
    PERSON_DF   = './mydailytravel/person.csv'
    HOUSEHOLD_DF = './mydailytravel/household.csv'
    LOCATION_DF = './mydailytravel/location.csv'

    # --- Spatial data sources ---
    # Built environment metrics on grid
    BE_DF = '/blue/shenhaowang/qingqisong/GenerativeUrbanDesign-main/metrics_datagridc180/metrics_results.csv'
    # Pre-computed transit proximity per household
    TRANSIT_FIXED_DF = './final_df_transit_fixed.csv'
    # CBD boundary shapefile
    CBD_SHP = './Central_Business_District_20260120/geo_export_48893cd7-ee67-4b97-afee-03338634fe3e.shp'
    # Chicago city boundary shapefile
    CHICAGO_BOUNDARY_SHP = './Boundaries_City_20260209/geo_export_8fd7b0b2-d957-4597-87c2-aa49c13e4eeb.shp'
    # Illinois census block group shapefile (for spatial zoning)
    CBG_SHP = './tl_2025_17_bg/tl_2025_17_bg.shp'
    # EPA Smart Location Database
    EPA_CSV = './epa_illinois_cleaned.csv'
    # Park facilities shapefile
    PARK_SHP = './CPD_Facilities_20260123/geo_export_35b9829f-ebed-4a35-bea8-8f8412132942.shp'

    # --- Output ---
    OUTPUT_OD_MATRIX = './chicago_od_flow_matrix.csv'
    OUTPUT_ZONE_FEATURES = './chicago_zone_features.csv'
    OUTPUT_OD_DATASET = './chicago_od_gravity_dataset.csv'

    # --- Coordinate Reference Systems ---
    CRS_WGS84 = 'EPSG:4326'
    CRS_ILLINOIS = 'EPSG:3435'  # Illinois State Plane East (feet)

    # --- Constants ---
    FEET_PER_MILE = 5280
    HOME_LOCNO = 10000

    # --- Mode codes (same as before) ---
    ACTIVE_MODES  = [101, 102, 103, 104]
    TRANSIT_MODES = [500, 501, 502, 503, 504, 505, 506, 509]

    # --- Built environment feature column names in the BE grid file ---
    BE_FEATURE_COLS = [
        'intersection_density',
        'road_network_complexity',
        'building_density',
        'land_use_diversity',
        'amenity_density'
    ]

    # --- EPA SLD columns to extract ---
    EPA_COLUMNS = ['D1B', 'D1C', 'D2A_EPHHM']

    # --- Chicago geographic bounds for coordinate validation ---
    LAT_MIN, LAT_MAX = 41.5, 42.5
    LON_MIN, LON_MAX = -88.5, -87.0


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def print_section(title: str):
    """Print a formatted section header."""
    print("\n" + "=" * 80)
    print(f"  {title}")
    print("=" * 80)


def print_step(step_num, description: str):
    """Print a formatted step header."""
    print(f"\n>>> STEP {step_num}: {description}")
    print("-" * 60)


def load_csv(filepath: str, description: str) -> pd.DataFrame:
    """Load a CSV file with validation and reporting."""
    print(f"  Loading {description}...")
    if not os.path.exists(filepath):
        print(f"    WARNING: File not found: {filepath}")
        return pd.DataFrame()
    df = pd.read_csv(filepath, low_memory=False)
    print(f"    Loaded: {len(df):,} rows x {len(df.columns)} columns")
    return df


def standardize_keys(df: pd.DataFrame, keys: list) -> pd.DataFrame:
    """Cast join-key columns to int64 for consistent merging."""
    for k in keys:
        if k in df.columns:
            df[k] = pd.to_numeric(df[k], errors='coerce').fillna(-999).astype(np.int64)
    return df


# ############################################################################
#
#   PHASE 1: EXTRACT TRIP-LEVEL ORIGIN-DESTINATION PAIRS
#
# ############################################################################

def extract_trip_od_pairs(place_df: pd.DataFrame,
                          location_df: pd.DataFrame,
                          valid_sampnos: set = None,
                          home_based_only: bool = True) -> pd.DataFrame:
    """
    Extract origin-destination pairs for every trip segment.

    For each trip in the place file:
      - Destination = current row's locno
      - Origin      = previous row's locno (first trip assumes home = 10000)

    Then look up lat/lon coordinates for both origin and destination
    from the location file.

    Parameters
    ----------
    place_df : pd.DataFrame
        Raw place (trip) file from My Daily Travel survey.
    location_df : pd.DataFrame
        Location file with geocoded coordinates for each locno.
    valid_sampnos : set or None
        If provided, restrict to these household IDs (e.g., Chicago only).
    home_based_only : bool
        If True, keep only home-based trips (origin or destination is home).

    Returns
    -------
    pd.DataFrame
        Trip-level data with columns:
        sampno, perno, trip_seq, mode, travtime,
        orig_locno, dest_locno,
        orig_lat, orig_lon, dest_lat, dest_lon,
        is_home_based
    """

    print("  Extracting trip-level OD pairs...")

    trips = place_df.copy()

    # --- Filter to Chicago residents if boundary is applied ---
    if valid_sampnos is not None:
        trips = trips[trips['sampno'].isin(valid_sampnos)].copy()
        print(f"    Filtered to Chicago residents: {len(trips):,} trip segments")

    # --- Coerce key columns ---
    trips['locno']    = pd.to_numeric(trips['locno'], errors='coerce')
    trips['travtime'] = pd.to_numeric(trips['travtime'], errors='coerce').fillna(0).clip(lower=0)

    # --- Sort trips within each person's day ---
    # 'plession' is the trip sequence number in My Daily Travel
    sort_col = 'plession' if 'plession' in trips.columns else 'placeno'
    trips = trips.sort_values(['sampno', 'perno', sort_col]).reset_index(drop=True)

    # --- Determine origin locno for each trip ---
    # Origin of trip n = destination of trip (n-1)
    # For the very first trip of each person, assume origin = home (10000)
    trips['dest_locno'] = trips['locno']
    trips['orig_locno'] = (
        trips.groupby(['sampno', 'perno'])['locno']
        .shift(1)
        .fillna(Config.HOME_LOCNO)
        .astype(int)
    )

    # --- Assign trip sequence number ---
    trips['trip_seq'] = trips.groupby(['sampno', 'perno']).cumcount() + 1

    # --- Identify home-based trips ---
    trips['is_home_orig'] = (trips['orig_locno'] == Config.HOME_LOCNO)
    trips['is_home_dest'] = (trips['dest_locno'] == Config.HOME_LOCNO)
    trips['is_home_based'] = trips['is_home_orig'] | trips['is_home_dest']

    total = len(trips)
    hb    = trips['is_home_based'].sum()
    print(f"    Total trips: {total:,}")
    print(f"    Home-based trips: {hb:,} ({hb/total*100:.1f}%)")

    # --- Filter to home-based only if requested ---
    if home_based_only:
        trips = trips[trips['is_home_based']].copy()
        print(f"    After HB filter: {len(trips):,} trips")

    # -----------------------------------------------------------------
    # Look up coordinates from the location file
    # -----------------------------------------------------------------
    # Build a lookup table: locno -> (lat, lon) per household
    loc = location_df.copy()
    loc['latitude']  = pd.to_numeric(loc['latitude'], errors='coerce')
    loc['longitude'] = pd.to_numeric(loc['longitude'], errors='coerce')

    # Keep only valid coordinates
    loc = loc.dropna(subset=['latitude', 'longitude'])
    loc = loc[
        (loc['latitude']  >= Config.LAT_MIN) & (loc['latitude']  <= Config.LAT_MAX) &
        (loc['longitude'] >= Config.LON_MIN) & (loc['longitude'] <= Config.LON_MAX)
    ]

    # Create lookup keyed on (sampno, locno)
    loc['sampno'] = pd.to_numeric(loc['sampno'], errors='coerce').astype('Int64')
    loc['locno']  = pd.to_numeric(loc['locno'],  errors='coerce').astype('Int64')
    loc_lookup = (
        loc.drop_duplicates(subset=['sampno', 'locno'], keep='first')
        [['sampno', 'locno', 'latitude', 'longitude']]
    )

    # --- Merge origin coordinates ---
    trips = trips.merge(
        loc_lookup.rename(columns={
            'locno': 'orig_locno',
            'latitude': 'orig_lat',
            'longitude': 'orig_lon'
        }),
        on=['sampno', 'orig_locno'],
        how='left'
    )

    # --- Merge destination coordinates ---
    trips = trips.merge(
        loc_lookup.rename(columns={
            'locno': 'dest_locno',
            'latitude': 'dest_lat',
            'longitude': 'dest_lon'
        }),
        on=['sampno', 'dest_locno'],
        how='left'
    )

    # --- Report coordinate match rates ---
    orig_matched = trips['orig_lat'].notna().sum()
    dest_matched = trips['dest_lat'].notna().sum()
    both_matched = (trips['orig_lat'].notna() & trips['dest_lat'].notna()).sum()
    print(f"    Origin coords matched:  {orig_matched:,} / {len(trips):,}")
    print(f"    Dest coords matched:    {dest_matched:,} / {len(trips):,}")
    print(f"    Both matched:           {both_matched:,} / {len(trips):,}")

    # --- Keep only trips with both O and D coordinates ---
    trips = trips.dropna(subset=['orig_lat', 'orig_lon', 'dest_lat', 'dest_lon'])
    print(f"    Trips with valid OD coords: {len(trips):,}")

    # --- Select output columns ---
    keep_cols = [
        'sampno', 'perno', 'trip_seq', 'mode', 'travtime',
        'orig_locno', 'dest_locno',
        'orig_lat', 'orig_lon', 'dest_lat', 'dest_lon',
        'is_home_based'
    ]
    keep_cols = [c for c in keep_cols if c in trips.columns]
    trips = trips[keep_cols].copy()

    print(f"\n  Phase 1 complete: {len(trips):,} trip OD pairs extracted")
    return trips


# ############################################################################
#
#   PHASE 2: ASSIGN ORIGINS AND DESTINATIONS TO CENSUS BLOCK GROUPS
#
# ############################################################################

def assign_trips_to_zones(trips_df: pd.DataFrame,
                          cbg_shp_path: str) -> pd.DataFrame:
    """
    Assign each trip's origin and destination to a Census Block Group
    using spatial point-in-polygon joins.

    Parameters
    ----------
    trips_df : pd.DataFrame
        Trip-level data with orig_lat/lon and dest_lat/lon columns.
    cbg_shp_path : str
        Path to the census block group shapefile.

    Returns
    -------
    pd.DataFrame
        Same trips with two new columns: orig_GEOID, dest_GEOID
    """

    print("  Assigning trips to Census Block Groups...")

    # --- Load block group boundaries ---
    cbg = gpd.read_file(cbg_shp_path)
    if cbg.crs is None:
        cbg = cbg.set_crs(Config.CRS_WGS84)
    cbg = cbg.to_crs(Config.CRS_ILLINOIS)

    # Identify the GEOID column
    geoid_col = None
    for candidate in ['GEOID', 'GEOID20', 'GEOID10']:
        if candidate in cbg.columns:
            geoid_col = candidate
            break
    if geoid_col is None:
        raise ValueError("No GEOID column found in block group shapefile")

    print(f"    Block groups loaded: {len(cbg):,}")
    print(f"    GEOID column: {geoid_col}")

    cbg_join = cbg[[geoid_col, 'geometry']].copy()

    # -----------------------------------------------------------------
    # 2a. Assign ORIGIN to CBG
    # -----------------------------------------------------------------
    print("    Assigning origins...")
    orig_pts = gpd.GeoDataFrame(
        trips_df,
        geometry=[Point(lon, lat) for lon, lat in
                  zip(trips_df['orig_lon'], trips_df['orig_lat'])],
        crs=Config.CRS_WGS84
    ).to_crs(Config.CRS_ILLINOIS)

    orig_joined = gpd.sjoin(
        orig_pts, cbg_join, how='left', predicate='within'
    )
    trips_df = trips_df.copy()
    trips_df['orig_GEOID'] = orig_joined[geoid_col].values

    # -----------------------------------------------------------------
    # 2b. Assign DESTINATION to CBG
    # -----------------------------------------------------------------
    print("    Assigning destinations...")
    dest_pts = gpd.GeoDataFrame(
        trips_df,
        geometry=[Point(lon, lat) for lon, lat in
                  zip(trips_df['dest_lon'], trips_df['dest_lat'])],
        crs=Config.CRS_WGS84
    ).to_crs(Config.CRS_ILLINOIS)

    dest_joined = gpd.sjoin(
        dest_pts, cbg_join, how='left', predicate='within'
    )
    trips_df['dest_GEOID'] = dest_joined[geoid_col].values

    # --- Report ---
    orig_ok = trips_df['orig_GEOID'].notna().sum()
    dest_ok = trips_df['dest_GEOID'].notna().sum()
    both_ok = (trips_df['orig_GEOID'].notna() & trips_df['dest_GEOID'].notna()).sum()
    print(f"    Origins assigned:  {orig_ok:,} / {len(trips_df):,}")
    print(f"    Destinations assigned: {dest_ok:,} / {len(trips_df):,}")
    print(f"    Both assigned:     {both_ok:,} / {len(trips_df):,}")

    # Drop trips that could not be assigned
    trips_df = trips_df.dropna(subset=['orig_GEOID', 'dest_GEOID']).copy()
    print(f"    Trips with valid OD zones: {len(trips_df):,}")

    return trips_df


# ############################################################################
#
#   PHASE 3: AGGREGATE INTO OD FLOW MATRIX
#
# ############################################################################

def build_od_matrix(trips_df: pd.DataFrame,
                    include_zeros: bool = True) -> pd.DataFrame:
    """
    Aggregate individual trips into an OD flow matrix.

    Each row in the output represents one OD pair (origin_CBG -> dest_CBG)
    with the total trip count as the dependent variable.

    Parameters
    ----------
    trips_df : pd.DataFrame
        Trip-level data with orig_GEOID and dest_GEOID columns.
    include_zeros : bool
        If True, add rows for zone pairs with zero observed trips.
        This is important for the gravity model to learn from "no flow"
        pairs as well as high-flow pairs.

    Returns
    -------
    pd.DataFrame
        Columns: orig_GEOID, dest_GEOID, trip_count
    """

    print("  Building OD flow matrix...")

    # --- Count trips for each OD pair ---
    od_counts = (
        trips_df.groupby(['orig_GEOID', 'dest_GEOID'])
        .size()
        .reset_index(name='trip_count')
    )

    n_pairs_nonzero = len(od_counts)
    total_trips = od_counts['trip_count'].sum()
    print(f"    Non-zero OD pairs: {n_pairs_nonzero:,}")
    print(f"    Total trips: {total_trips:,}")

    # --- Optionally include zero-flow pairs ---
    if include_zeros:
        all_zones = sorted(
            set(trips_df['orig_GEOID'].unique()) |
            set(trips_df['dest_GEOID'].unique())
        )
        n_zones = len(all_zones)
        print(f"    Unique zones: {n_zones:,}")
        print(f"    Potential OD pairs (incl. intra-zone): {n_zones * n_zones:,}")

        # Create full cross-product of zones
        from itertools import product
        full_od = pd.DataFrame(
            list(product(all_zones, all_zones)),
            columns=['orig_GEOID', 'dest_GEOID']
        )

        # Optionally remove intra-zone (self-loop) pairs
        # For a gravity model, intra-zone flows are often excluded
        # because distance = 0 causes problems with log(distance)
        full_od = full_od[full_od['orig_GEOID'] != full_od['dest_GEOID']].copy()
        print(f"    Inter-zone OD pairs: {len(full_od):,}")

        # Left merge to get counts (fill missing with 0)
        od_matrix = full_od.merge(od_counts, on=['orig_GEOID', 'dest_GEOID'], how='left')
        od_matrix['trip_count'] = od_matrix['trip_count'].fillna(0).astype(int)
    else:
        od_matrix = od_counts.copy()

    # --- Summary statistics ---
    n_zero = (od_matrix['trip_count'] == 0).sum()
    n_nonzero = (od_matrix['trip_count'] > 0).sum()
    sparsity = n_zero / len(od_matrix) * 100

    print(f"\n    OD Matrix Summary:")
    print(f"      Total OD pairs:   {len(od_matrix):,}")
    print(f"      Non-zero pairs:   {n_nonzero:,}")
    print(f"      Zero pairs:       {n_zero:,}")
    print(f"      Sparsity:         {sparsity:.1f}%")
    print(f"      Mean flow:        {od_matrix['trip_count'].mean():.4f}")
    print(f"      Max flow:         {od_matrix['trip_count'].max():,}")

    return od_matrix


# ############################################################################
#
#   PHASE 4: COMPUTE ZONE-LEVEL FEATURES
#
# ############################################################################

def compute_zone_features(cbg_shp_path: str,
                          epa_csv_path: str,
                          be_csv_path: str,
                          cbd_shp_path: str,
                          park_shp_path: str,
                          relevant_geoids: set) -> pd.DataFrame:
    """
    Compute a comprehensive set of features for each census block group.

    These features will serve as both origin and destination attributes
    in the gravity model.

    Parameters
    ----------
    cbg_shp_path : str
        Path to census block group shapefile (for geometry / centroids).
    epa_csv_path : str
        Path to EPA Smart Location Database CSV.
    be_csv_path : str
        Path to grid built environment metrics.
    cbd_shp_path : str
        Path to CBD boundary shapefile.
    park_shp_path : str
        Path to park facilities shapefile.
    relevant_geoids : set
        Set of GEOIDs that appear in the OD matrix (to limit computation).

    Returns
    -------
    pd.DataFrame
        One row per CBG with columns: GEOID, centroid_x, centroid_y,
        D1B, D1C, D2A_EPHHM, avg_intersection_density, ..., dist_cbd_mi,
        dist_park_mi
    """

    print("  Computing zone-level features...")

    # -----------------------------------------------------------------
    # 4.1  Load block group geometries and compute centroids
    # -----------------------------------------------------------------
    print("    [4.1] Loading block group geometries & centroids...")
    cbg = gpd.read_file(cbg_shp_path)
    if cbg.crs is None:
        cbg = cbg.set_crs(Config.CRS_WGS84)
    cbg = cbg.to_crs(Config.CRS_ILLINOIS)

    # Find GEOID column
    geoid_col = next(c for c in ['GEOID', 'GEOID20', 'GEOID10'] if c in cbg.columns)
    cbg = cbg.rename(columns={geoid_col: 'GEOID'})

    # Filter to relevant zones only
    cbg = cbg[cbg['GEOID'].isin(relevant_geoids)].copy()
    print(f"      Relevant zones: {len(cbg):,}")

    # Compute centroids (in projected CRS, units = feet)
    cbg['centroid_x'] = cbg.geometry.centroid.x
    cbg['centroid_y'] = cbg.geometry.centroid.y

    zone_features = cbg[['GEOID', 'centroid_x', 'centroid_y']].copy()

    # -----------------------------------------------------------------
    # 4.2  EPA Smart Location Database (population & employment density)
    # -----------------------------------------------------------------
    print("    [4.2] Merging EPA SLD features...")
    if os.path.exists(epa_csv_path):
        epa = pd.read_csv(epa_csv_path, low_memory=False)

        # Find GEOID column in EPA
        epa_gid = next((c for c in ['GEOID', 'GEOID20', 'GEOID10'] if c in epa.columns), None)
        if epa_gid:
            epa['GEOID'] = epa[epa_gid].astype(str).str.replace('.0', '', regex=False).str.zfill(12)
            zone_features['GEOID'] = zone_features['GEOID'].astype(str).str.zfill(12)

            epa_cols = ['GEOID'] + [c for c in Config.EPA_COLUMNS if c in epa.columns]
            epa_sub = epa[epa_cols].drop_duplicates(subset='GEOID', keep='first')

            zone_features = zone_features.merge(epa_sub, on='GEOID', how='left')

            for col in Config.EPA_COLUMNS:
                if col in zone_features.columns:
                    matched = zone_features[col].notna().sum()
                    print(f"      {col}: {matched:,} matched")
    else:
        print(f"      WARNING: EPA file not found")
        for col in Config.EPA_COLUMNS:
            zone_features[col] = np.nan

    # -----------------------------------------------------------------
    # 4.3  Built environment metrics (average grid values per zone)
    # -----------------------------------------------------------------
    print("    [4.3] Computing average BE metrics per zone...")

    if os.path.exists(be_csv_path):
        be = pd.read_csv(be_csv_path)
        be['lat'] = pd.to_numeric(be['lat'], errors='coerce')
        be['lon'] = pd.to_numeric(be['lon'], errors='coerce')
        be = be.dropna(subset=['lat', 'lon'])

        available_be = [c for c in Config.BE_FEATURE_COLS if c in be.columns]

        if available_be:
            # Create GeoDataFrame for BE grid points
            be_gdf = gpd.GeoDataFrame(
                be,
                geometry=[Point(lon, lat) for lon, lat in zip(be['lon'], be['lat'])],
                crs=Config.CRS_WGS84
            ).to_crs(Config.CRS_ILLINOIS)

            # Spatial join: assign each grid point to a CBG
            cbg_for_join = cbg[['GEOID', 'geometry']].copy()
            cbg_for_join['GEOID'] = cbg_for_join['GEOID'].astype(str).str.zfill(12)

            be_in_cbg = gpd.sjoin(be_gdf, cbg_for_join, how='inner', predicate='within')

            # Average BE metrics per zone
            zone_be = be_in_cbg.groupby('GEOID')[available_be].mean().reset_index()
            zone_be.columns = ['GEOID'] + [f'avg_{c}' for c in available_be]

            zone_features = zone_features.merge(zone_be, on='GEOID', how='left')
            print(f"      BE metrics averaged for {len(zone_be):,} zones")
    else:
        print(f"      WARNING: BE file not found")

    # -----------------------------------------------------------------
    # 4.4  Distance to CBD (from zone centroid to CBD boundary)
    # -----------------------------------------------------------------
    print("    [4.4] Computing distance to CBD...")

    if os.path.exists(cbd_shp_path):
        cbd_gdf = gpd.read_file(cbd_shp_path)
        if cbd_gdf.crs is None:
            cbd_gdf = cbd_gdf.set_crs(Config.CRS_WGS84)
        cbd_gdf = cbd_gdf.to_crs(Config.CRS_ILLINOIS)
        cbd_boundary = cbd_gdf.geometry.unary_union

        # Compute distance from each zone centroid to CBD boundary
        centroids = [
            Point(x, y) for x, y in
            zip(zone_features['centroid_x'], zone_features['centroid_y'])
        ]
        zone_features['dist_cbd_ft'] = [
            0 if cbd_boundary.contains(pt) else pt.distance(cbd_boundary.boundary)
            for pt in centroids
        ]
        zone_features['dist_cbd_mi'] = zone_features['dist_cbd_ft'] / Config.FEET_PER_MILE
        print(f"      Mean dist to CBD: {zone_features['dist_cbd_mi'].mean():.2f} mi")
    else:
        print(f"      WARNING: CBD shapefile not found")
        zone_features['dist_cbd_mi'] = np.nan

    # -----------------------------------------------------------------
    # 4.5  Distance to nearest park (from zone centroid)
    # -----------------------------------------------------------------
    print("    [4.5] Computing distance to nearest park...")

    if os.path.exists(park_shp_path):
        parks = gpd.read_file(park_shp_path)
        if parks.crs is None:
            parks = parks.set_crs(Config.CRS_WGS84)
        parks = parks.to_crs(Config.CRS_ILLINOIS)

        park_coords = np.column_stack([parks.geometry.x, parks.geometry.y])
        park_tree = KDTree(park_coords)

        centroid_coords = np.column_stack([
            zone_features['centroid_x'], zone_features['centroid_y']
        ])
        dists, _ = park_tree.query(centroid_coords)
        zone_features['dist_park_ft'] = dists
        zone_features['dist_park_mi'] = dists / Config.FEET_PER_MILE
        print(f"      Mean dist to park: {zone_features['dist_park_mi'].mean():.2f} mi")
    else:
        print(f"      WARNING: Park shapefile not found")
        zone_features['dist_park_mi'] = np.nan

    print(f"\n  Phase 4 complete: {len(zone_features):,} zones with features")
    return zone_features


# ############################################################################
#
#   PHASE 5: BUILD OD-PAIR FEATURE DATASET
#
# ############################################################################

def build_od_feature_dataset(od_matrix: pd.DataFrame,
                             zone_features: pd.DataFrame) -> pd.DataFrame:
    """
    Construct the final OD-pair feature dataset for the gravity model.

    For each OD pair, attach:
      - Origin zone features (prefixed with O_)
      - Destination zone features (prefixed with D_)
      - Centroid-to-centroid distance in miles

    Parameters
    ----------
    od_matrix : pd.DataFrame
        Columns: orig_GEOID, dest_GEOID, trip_count
    zone_features : pd.DataFrame
        Columns: GEOID, centroid_x, centroid_y, D1B, D1C, ..., etc.

    Returns
    -------
    pd.DataFrame
        Complete OD-pair feature dataset ready for modeling.
    """

    print("  Building OD-pair feature dataset...")

    # Identify feature columns (everything except GEOID and centroid coords)
    meta_cols = ['GEOID', 'centroid_x', 'centroid_y']
    feat_cols = [c for c in zone_features.columns if c not in meta_cols]

    # -----------------------------------------------------------------
    # 5.1  Merge origin zone features
    # -----------------------------------------------------------------
    origin_feats = zone_features.copy()
    origin_rename = {c: f'O_{c}' for c in feat_cols}
    origin_rename['centroid_x'] = 'O_centroid_x'
    origin_rename['centroid_y'] = 'O_centroid_y'
    origin_feats = origin_feats.rename(columns=origin_rename)
    origin_feats = origin_feats.rename(columns={'GEOID': 'orig_GEOID'})

    od_data = od_matrix.merge(origin_feats, on='orig_GEOID', how='left')
    print(f"    After origin merge: {len(od_data):,} rows")

    # -----------------------------------------------------------------
    # 5.2  Merge destination zone features
    # -----------------------------------------------------------------
    dest_feats = zone_features.copy()
    dest_rename = {c: f'D_{c}' for c in feat_cols}
    dest_rename['centroid_x'] = 'D_centroid_x'
    dest_rename['centroid_y'] = 'D_centroid_y'
    dest_feats = dest_feats.rename(columns=dest_rename)
    dest_feats = dest_feats.rename(columns={'GEOID': 'dest_GEOID'})

    od_data = od_data.merge(dest_feats, on='dest_GEOID', how='left')
    print(f"    After destination merge: {len(od_data):,} rows")

    # -----------------------------------------------------------------
    # 5.3  Compute inter-zone distance (centroid to centroid)
    # -----------------------------------------------------------------
    print("    Computing inter-zone distances...")

    # Distance in feet (since CRS is EPSG:3435, units = feet)
    od_data['dist_od_ft'] = np.sqrt(
        (od_data['O_centroid_x'] - od_data['D_centroid_x'])**2 +
        (od_data['O_centroid_y'] - od_data['D_centroid_y'])**2
    )
    od_data['dist_od_mi'] = od_data['dist_od_ft'] / Config.FEET_PER_MILE

    # Log-distance (used in gravity model; add small epsilon to avoid log(0))
    od_data['log_dist_mi'] = np.log(od_data['dist_od_mi'].clip(lower=0.01))

    print(f"      Mean OD distance: {od_data['dist_od_mi'].mean():.2f} mi")
    print(f"      Median OD distance: {od_data['dist_od_mi'].median():.2f} mi")

    # -----------------------------------------------------------------
    # 5.4  Clean up: drop coordinate columns, keep features
    # -----------------------------------------------------------------
    drop_cols = ['O_centroid_x', 'O_centroid_y', 'D_centroid_x', 'D_centroid_y']
    od_data = od_data.drop(columns=[c for c in drop_cols if c in od_data.columns])

    print(f"\n  Phase 5 complete: {len(od_data):,} OD pairs, {len(od_data.columns)} columns")
    return od_data


# ############################################################################
#
#   PHASE 6: FIT BASELINE GRAVITY MODEL
#
# ############################################################################

def fit_gravity_model(od_data: pd.DataFrame):
    """
    Fit baseline gravity models and evaluate performance.

    Model 1 (Basic Gravity):
        ln(T_ij) ~ ln(O_pop) + ln(D_pop) + ln(distance)

    Model 2 (Enhanced Gravity):
        ln(T_ij) ~ all origin features + all dest features + ln(distance)

    Both are fitted using Poisson regression.

    Parameters
    ----------
    od_data : pd.DataFrame
        OD-pair feature dataset from Phase 5.

    Returns
    -------
    dict
        Dictionary with model results and evaluation metrics.
    """

    import statsmodels.api as sm
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_squared_error, mean_absolute_error

    print("  Fitting gravity models...")

    df = od_data.copy()

    # --- Prepare log-transformed population variables ---
    # Use D1B (gross population density) as the "mass" variable
    # Add small epsilon to avoid log(0)
    eps = 0.01
    if 'O_D1B' in df.columns and 'D_D1B' in df.columns:
        df['log_O_pop'] = np.log(df['O_D1B'].clip(lower=eps))
        df['log_D_pop'] = np.log(df['D_D1B'].clip(lower=eps))
    else:
        print("    WARNING: D1B not available, using placeholder")
        df['log_O_pop'] = 0
        df['log_D_pop'] = 0

    # --- Drop rows with NaN in key columns ---
    model_cols_basic = ['log_O_pop', 'log_D_pop', 'log_dist_mi']
    df_clean = df.dropna(subset=model_cols_basic + ['trip_count'])
    print(f"    Clean rows for modeling: {len(df_clean):,}")

    # --- Train/test split (80/20) ---
    train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
    print(f"    Train: {len(train_df):,} | Test: {len(test_df):,}")

    results = {}

    # =================================================================
    # MODEL 1: Basic Gravity (population + distance only)
    # =================================================================
    print("\n    --- Model 1: Basic Gravity ---")

    X_train_basic = sm.add_constant(train_df[model_cols_basic])
    X_test_basic  = sm.add_constant(test_df[model_cols_basic])
    y_train = train_df['trip_count']
    y_test  = test_df['trip_count']

    try:
        poisson_basic = sm.GLM(
            y_train, X_train_basic,
            family=sm.families.Poisson()
        ).fit()

        print(poisson_basic.summary().tables[1])

        # Predict
        y_pred_basic = poisson_basic.predict(X_test_basic)

        # Evaluate
        cpc_basic = compute_cpc(y_test.values, y_pred_basic.values)
        rmse_basic = np.sqrt(mean_squared_error(y_test, y_pred_basic))
        mae_basic = mean_absolute_error(y_test, y_pred_basic)
        corr_basic = np.corrcoef(y_test, y_pred_basic)[0, 1]

        print(f"\n    Basic Gravity Results:")
        print(f"      CPC:         {cpc_basic:.4f}")
        print(f"      RMSE:        {rmse_basic:.4f}")
        print(f"      MAE:         {mae_basic:.4f}")
        print(f"      Correlation: {corr_basic:.4f}")

        results['basic'] = {
            'model': poisson_basic,
            'CPC': cpc_basic, 'RMSE': rmse_basic,
            'MAE': mae_basic, 'Corr': corr_basic
        }
    except Exception as e:
        print(f"    Basic model failed: {e}")

    # =================================================================
    # MODEL 2: Enhanced Gravity (all zone features + distance)
    # =================================================================
    print("\n    --- Model 2: Enhanced Gravity ---")

    # Identify all available feature columns
    o_feat_cols = [c for c in df_clean.columns if c.startswith('O_') and c != 'O_centroid_x' and c != 'O_centroid_y']
    d_feat_cols = [c for c in df_clean.columns if c.startswith('D_') and c != 'D_centroid_x' and c != 'D_centroid_y']
    all_feat_cols = o_feat_cols + d_feat_cols + ['log_dist_mi']

    # Drop columns with too many NaN
    valid_feat_cols = [
        c for c in all_feat_cols
        if df_clean[c].notna().sum() > len(df_clean) * 0.5
    ]

    df_enhanced = df_clean.dropna(subset=valid_feat_cols + ['trip_count'])
    print(f"      Features: {len(valid_feat_cols)}")
    print(f"      Clean rows: {len(df_enhanced):,}")

    if len(df_enhanced) > 100:
        train_e, test_e = train_test_split(df_enhanced, test_size=0.2, random_state=42)

        X_train_enh = sm.add_constant(train_e[valid_feat_cols].astype(float))
        X_test_enh  = sm.add_constant(test_e[valid_feat_cols].astype(float))
        y_train_e = train_e['trip_count']
        y_test_e  = test_e['trip_count']

        try:
            poisson_enh = sm.GLM(
                y_train_e, X_train_enh,
                family=sm.families.Poisson()
            ).fit()

            y_pred_enh = poisson_enh.predict(X_test_enh)

            cpc_enh  = compute_cpc(y_test_e.values, y_pred_enh.values)
            rmse_enh = np.sqrt(mean_squared_error(y_test_e, y_pred_enh))
            mae_enh  = mean_absolute_error(y_test_e, y_pred_enh)
            corr_enh = np.corrcoef(y_test_e, y_pred_enh)[0, 1]

            print(f"\n    Enhanced Gravity Results:")
            print(f"      CPC:         {cpc_enh:.4f}")
            print(f"      RMSE:        {rmse_enh:.4f}")
            print(f"      MAE:         {mae_enh:.4f}")
            print(f"      Correlation: {corr_enh:.4f}")

            results['enhanced'] = {
                'model': poisson_enh,
                'CPC': cpc_enh, 'RMSE': rmse_enh,
                'MAE': mae_enh, 'Corr': corr_enh
            }
        except Exception as e:
            print(f"    Enhanced model failed: {e}")

    return results


def compute_cpc(y_real: np.ndarray, y_gen: np.ndarray) -> float:
    """
    Compute Common Part of Commuters (CPC).

    CPC = 2 * sum(min(y_real, y_gen)) / (sum(y_real) + sum(y_gen))

    This is the standard metric for evaluating gravity models,
    as used in both the Deep Gravity and ZINB-GNN papers.

    Ranges from 0 (no overlap) to 1 (perfect match).
    """
    numerator   = 2.0 * np.sum(np.minimum(y_real, y_gen))
    denominator = np.sum(y_real) + np.sum(y_gen)
    if denominator == 0:
        return 0.0
    return numerator / denominator


# ############################################################################
#
#   MAIN PIPELINE
#
#
# ############################################################################

def run_gravity_pipeline():
    """Execute the complete gravity model pipeline."""

    print_section("CHICAGO BASELINE GRAVITY MODEL FOR OD PREDICTION")
    print(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    # =====================================================================
    # STEP 1: Load raw data
    # =====================================================================
    print_step(1, "Loading Data Sources")

    place_df     = load_csv(Config.PLACE_DF,     "Place/Trip Data")
    person_df    = load_csv(Config.PERSON_DF,     "Person Data")
    household_df = load_csv(Config.HOUSEHOLD_DF,  "Household Data")
    location_df  = load_csv(Config.LOCATION_DF,   "Location Data")

    # =====================================================================
    # STEP 2: Filter to Chicago city boundary (FIXED)
    # =====================================================================
    print_step(2, "Filtering to Chicago City Boundary")

    chicago_boundary = None
    valid_sampnos = None

    if os.path.exists(Config.CHICAGO_BOUNDARY_SHP):
        chi_gdf = gpd.read_file(Config.CHICAGO_BOUNDARY_SHP)
        if chi_gdf.crs is None:
            chi_gdf = chi_gdf.set_crs(Config.CRS_WGS84)
        chi_gdf = chi_gdf.to_crs(Config.CRS_ILLINOIS)
        chicago_boundary = chi_gdf.geometry.unary_union

        # ---------------------------------------------------------
        # FIX: loctype is int64, not string.
        # Use home==1 to filter home locations.
        # ---------------------------------------------------------
        loc = location_df.copy()
        print(f"  Location file: {len(loc):,} rows")
        print(f"  loctype dtype: {loc['loctype'].dtype}")
        print(f"  home dtype: {loc['home'].dtype}")

        # Filter for home locations using integer column
        homes = loc[loc['home'] == 1].copy()
        print(f"  Home locations (home==1): {len(homes):,}")

        if len(homes) == 0:
            print("  WARNING: No home locations found!")
        else:
            # Clean coordinates
            homes['latitude']  = pd.to_numeric(homes['latitude'], errors='coerce')
            homes['longitude'] = pd.to_numeric(homes['longitude'], errors='coerce')
            homes = homes.dropna(subset=['latitude', 'longitude'])

            # Validate coordinate range
            homes = homes[
                (homes['latitude']  >= Config.LAT_MIN) &
                (homes['latitude']  <= Config.LAT_MAX) &
                (homes['longitude'] >= Config.LON_MIN) &
                (homes['longitude'] <= Config.LON_MAX)
            ].copy()
            print(f"  Valid coordinates: {len(homes):,}")

            # One record per household
            homes = homes.drop_duplicates(subset='sampno', keep='first')
            print(f"  Unique households: {len(homes):,}")

            # Create GeoDataFrame
            geom = [Point(lon, lat) for lon, lat in
                    zip(homes['longitude'], homes['latitude'])]
            homes_gdf = gpd.GeoDataFrame(
                homes, geometry=geom, crs=Config.CRS_WGS84
            )
            homes_gdf = homes_gdf.to_crs(Config.CRS_ILLINOIS)

            # Check which homes are inside Chicago
            homes_gdf['in_chicago'] = homes_gdf.geometry.apply(
                lambda pt: chicago_boundary.contains(pt)
            )

            valid_sampnos = set(
                homes_gdf.loc[homes_gdf['in_chicago'], 'sampno'].unique()
            )
            n_outside = (~homes_gdf['in_chicago']).sum()
            print(f"  Households inside Chicago:  {len(valid_sampnos):,}")
            print(f"  Households outside Chicago: {n_outside:,}")

    else:
        print("  WARNING: Chicago boundary not found, using all data")

    # =====================================================================
    # STEP 3: Extract trip-level OD pairs
    # =====================================================================
    print_step(3, "Extracting Trip-Level OD Pairs (Phase 1)")

    trips_od = extract_trip_od_pairs(
        place_df, location_df,
        valid_sampnos=valid_sampnos,
        home_based_only=True   # <-- set to False for ALL trips
    )

    # =====================================================================
    # STEP 4: Assign O/D to Census Block Groups
    # =====================================================================
    print_step(4, "Assigning Trips to Census Block Groups (Phase 2)")

    trips_with_zones = assign_trips_to_zones(trips_od, Config.CBG_SHP)

    # =====================================================================
    # STEP 5: Build OD flow matrix
    # =====================================================================
    print_step(5, "Building OD Flow Matrix (Phase 3)")

    od_matrix = build_od_matrix(trips_with_zones, include_zeros=True)

    # Save OD matrix
    od_matrix.to_csv(Config.OUTPUT_OD_MATRIX, index=False)
    print(f"  Saved: {Config.OUTPUT_OD_MATRIX}")

    # =====================================================================
    # STEP 6: Compute zone-level features
    # =====================================================================
    print_step(6, "Computing Zone-Level Features (Phase 4)")

    all_geoids = set(od_matrix['orig_GEOID'].unique()) | set(od_matrix['dest_GEOID'].unique())

    zone_features = compute_zone_features(
        cbg_shp_path  = Config.CBG_SHP,
        epa_csv_path  = Config.EPA_CSV,
        be_csv_path   = Config.BE_DF,
        cbd_shp_path  = Config.CBD_SHP,
        park_shp_path = Config.PARK_SHP,
        relevant_geoids = all_geoids
    )

    # Save zone features
    zone_features.to_csv(Config.OUTPUT_ZONE_FEATURES, index=False)
    print(f"  Saved: {Config.OUTPUT_ZONE_FEATURES}")

    # =====================================================================
    # STEP 7: Build OD-pair feature dataset
    # =====================================================================
    print_step(7, "Building OD-Pair Feature Dataset (Phase 5)")

    od_dataset = build_od_feature_dataset(od_matrix, zone_features)

    # Save OD feature dataset
    od_dataset.to_csv(Config.OUTPUT_OD_DATASET, index=False)
    print(f"  Saved: {Config.OUTPUT_OD_DATASET}")

    # =====================================================================
    # STEP 8: Fit gravity model and evaluate
    # =====================================================================
    print_step(8, "Fitting Gravity Models (Phase 6)")

    model_results = fit_gravity_model(od_dataset)

    # =====================================================================
    # FINAL SUMMARY
    # =====================================================================
    print_section("PIPELINE COMPLETE")
    print(f"""
    Dataset Summary:
      Trips extracted:     {len(trips_od):,}
      Zones (CBGs):        {len(all_geoids):,}
      OD pairs:            {len(od_matrix):,}
      Non-zero OD pairs:   {(od_matrix['trip_count'] > 0).sum():,}
      Sparsity:            {(od_matrix['trip_count'] == 0).sum() / len(od_matrix) * 100:.1f}%
    """)

    if 'basic' in model_results:
        r = model_results['basic']
        print(f"    Basic Gravity:    CPC={r['CPC']:.4f}  RMSE={r['RMSE']:.4f}  Corr={r['Corr']:.4f}")
    if 'enhanced' in model_results:
        r = model_results['enhanced']
        print(f"    Enhanced Gravity: CPC={r['CPC']:.4f}  RMSE={r['RMSE']:.4f}  Corr={r['Corr']:.4f}")

    return od_dataset, model_results


# ============================================================================
# ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    try:
        od_dataset, results = run_gravity_pipeline()
    except Exception as e:
        print(f"\nERROR: {e}")
        import traceback
        traceback.print_exc()



  CHICAGO BASELINE GRAVITY MODEL FOR OD PREDICTION
  Timestamp: 2026-03-10 18:15:51

>>> STEP 1: Loading Data Sources
------------------------------------------------------------
  Loading Place/Trip Data...
    Loaded: 77,313 rows x 105 columns
  Loading Person Data...
    Loaded: 30,683 rows x 116 columns
  Loading Household Data...
    Loaded: 12,391 rows x 25 columns
  Loading Location Data...
    Loaded: 110,091 rows x 13 columns

>>> STEP 2: Filtering to Chicago City Boundary
------------------------------------------------------------
  Location file: 110,091 rows
  loctype dtype: int64
  home dtype: int64
  Home locations (home==1): 12,747
  Valid coordinates: 12,139
  Unique households: 11,825
  Households inside Chicago:  4,550
  Households outside Chicago: 7,275

>>> STEP 3: Extracting Trip-Level OD Pairs (Phase 1)
------------------------------------------------------------
  Extracting trip-level OD pairs...
    Filtered to Chicago residents: 28,448 trip segments
    Tota